# Deep Learning Course - Module 4
# Convolutional Neural Networks (CNN)

Welcome to **Module 4** of the Deep Learning course! In this comprehensive notebook, we explore **Convolutional Neural Networks (CNNs)**—the foundational architecture behind modern computer vision.

---

## 📋 Table of Contents
1. [CNN Fundamentals](#1-cnn-fundamentals)
   - [Why CNNs over MLPs for Visual Data?](#why-cnns-over-mlps-for-visual-data)
   - [2D Convolution Operation from Scratch (NumPy)](#2d-convolution-operation-from-scratch-numpy)
   - [Padding and Stride](#padding-and-stride)
   - [Pooling Operations (Max & Average Pooling)](#pooling-operations-max--average-pooling)
   - [Visualizing Convolution Filters on Images](#visualizing-convolution-filters-on-images)
2. [Building CNNs with TensorFlow/Keras](#2-building-cnns-with-tensorflowkeras)
   - [Keras Building Blocks: Conv2D, Pooling, Flatten, Dense](#keras-building-blocks)
   - [Dataset Preparation: CIFAR-10](#dataset-preparation-cifar-10)
   - [Designing a Multi-Layer CNN Architecture](#designing-a-multi-layer-cnn-architecture)
   - [Training, Optimization & Loss/Accuracy Plotting](#training-optimization--lossaccuracy-plotting)
   - [Model Evaluation & Confusion Matrix](#model-evaluation--confusion-matrix)
3. [Popular CNN Architectures](#3-popular-cnn-architectures)
   - [LeNet-5 Implementation](#lenet-5-implementation)
   - [VGG Architecture (VGG-16 / VGG-19)](#vgg-architecture)
   - [ResNet Architecture & Residual Connections (Skip Connections)](#resnet-architecture--residual-connections)
4. [Transfer Learning](#4-transfer-learning)
   - [Feature Extraction vs. Fine-Tuning](#feature-extraction-vs-fine-tuning)
   - [Using Pre-trained Models (VGG16 / ResNet50)](#using-pre-trained-models)
   - [Fine-Tuning on Custom Data](#fine-tuning-on-custom-data)
   - [Performance Comparison](#performance-comparison)
5. [CNN Visualization & Interpretability](#5-cnn-visualization--interpretability)
   - [Visualizing Intermediate Layer Activations (Feature Maps)](#visualizing-intermediate-layer-activations)
   - [Visualizing Convolutional Filters (Kernels)](#visualizing-convolutional-filters)
   - [Grad-CAM (Gradient-weighted Class Activation Mapping)](#grad-cam)
6. [Summary & Key Takeaways](#6-summary--key-takeaways)


import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, datasets, optimizers, callbacks
from sklearn.metrics import classification_report, confusion_matrix

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set plotting styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {getattr(keras, '__version__', tf.__version__)}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, datasets, optimizers, callbacks
from sklearn.metrics import classification_report, confusion_matrix

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set plotting styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")


---
<a id="1-cnn-fundamentals"></a>
# 1. CNN Fundamentals

### Why CNNs over MLPs for Visual Data?
Standard Multi-Layer Perceptrons (MLPs) treat images as 1D flattened vectors of numbers. This presents critical challenges:
1. **Explosion of Parameters**: A $256 \times 256 \times 3$ color image has $196,608$ inputs. A single hidden layer with $1,000$ neurons requires ~196 million parameters!
2. **Loss of Spatial Structure**: Flattening an image destroys 2D spatial relationships between adjacent pixels.
3. **Lack of Translation Invariance**: An object shifted slightly in the image yields completely different input vector indices, forcing an MLP to re-learn the pattern at every position.

**Convolutional Neural Networks (CNNs)** solve these issues using three core principles:
- **Local Receptive Fields**: Neurons connect only to small local regions of the input image.
- **Shared Weights (Parameter Sharing)**: The same set of weights (kernel/filter) slides across the entire image, detecting the same feature everywhere.
- **Spatial Subsampling (Pooling)**: Reducing resolution to achieve translation invariance and compact representations.

---
### Mathematical Formulation of 2D Convolution

For an input image $I$ and a $2D$ kernel $K$ of size $k_H \times k_W$:

$$O(i, j) = (I * K)(i, j) = \sum_{m=0}^{k_H-1} \sum_{n=0}^{k_W-1} I(i+m, j+n) \cdot K(m, n) + b$$

Where:
- $I(i+m, j+n)$ is the pixel value at spatial location $(i+m, j+n)$.
- $K(m, n)$ is the filter weight at position $(m, n)$.
- $b$ is the scalar bias term.

#### Spatial Dimension Formula:
For an input of height $H$ and width $W$, kernel size $K$, padding $P$, and stride $S$:

$$H_{\text{out}} = \left\lfloor \frac{H - K + 2P}{S} \right\rfloor + 1$$

$$W_{\text{out}} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$


In [ ]:
def conv2d_scratch(image, kernel, stride=1, padding='valid', bias=0.0):
    """
    Implements a 2D convolution operation from scratch using pure NumPy.
    
    Parameters:
    -----------
    image : numpy.ndarray
        2D or 3D input image array of shape (H, W) or (H, W, C_in).
    kernel : numpy.ndarray
        2D or 4D kernel weights of shape (k_H, k_W) or (k_H, k_W, C_in, C_out).
    stride : int
        Step size when sliding the kernel across the image.
    padding : str
        'valid' (no padding) or 'same' (zero-padding to retain spatial dimensions if stride=1).
    bias : float or numpy.ndarray
        Bias term added to output feature map.
        
    Returns:
    --------
    numpy.ndarray
        Filtered output feature map.
    """
    # Ensure 3D image shape (H, W, C_in)
    if image.ndim == 2:
        image = image[:, :, np.newaxis]
    # Ensure 4D kernel shape (k_H, k_W, C_in, C_out)
    if kernel.ndim == 2:
        kernel = kernel[:, :, np.newaxis, np.newaxis]
        
    H, W, C_in = image.shape
    k_H, k_W, k_C_in, C_out = kernel.shape
    assert C_in == k_C_in, f"Channel mismatch: image has {C_in}, kernel has {k_C_in}"
    
    # Compute padding
    if padding == 'same':
        pad_h = max((H - 1) * stride + k_H - H, 0)
        pad_w = max((W - 1) * stride + k_W - W, 0)
        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left
    elif padding == 'valid':
        pad_top = pad_bottom = pad_left = pad_right = 0
    else:
        raise ValueError("Padding must be either 'valid' or 'same'")
        
    # Apply zero padding
    padded_img = np.pad(
        image, 
        ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), 
        mode='constant', 
        constant_values=0
    )
    
    # Calculate output dimensions
    out_H = (H + pad_top + pad_bottom - k_H) // stride + 1
    out_W = (W + pad_left + pad_right - k_W) // stride + 1
    
    output = np.zeros((out_H, out_W, C_out))
    
    # Perform element-wise multiplication and summation
    for c_out in range(C_out):
        b = bias[c_out] if isinstance(bias, np.ndarray) else bias
        for i in range(out_H):
            for j in range(out_W):
                h_start = i * stride
                h_end = h_start + k_H
                w_start = j * stride
                w_end = w_start + k_W
                
                patch = padded_img[h_start:h_end, w_start:w_end, :]
                output[i, j, c_out] = np.sum(patch * kernel[:, :, :, c_out]) + b
                
    return output.squeeze()

# Verification on dummy 5x5 matrix
sample_img = np.arange(1, 26, dtype=np.float32).reshape(5, 5)
sample_kernel = np.array([[1, 0], [0, -1]], dtype=np.float32)

out_valid = conv2d_scratch(sample_img, sample_kernel, stride=1, padding='valid')
out_same = conv2d_scratch(sample_img, sample_kernel, stride=1, padding='same')

print("Input 5x5 Matrix:\n", sample_img)
print("\nKernel 2x2:\n", sample_kernel)
print("\nValid Conv Output Shape:", out_valid.shape)
print("Valid Conv Output:\n", out_valid)
print("\nSame Conv Output Shape:", out_same.shape)


### Padding & Pooling Operations

#### 1. Padding ('valid' vs 'same')
- **Valid Padding ($P = 0$)**: No zero-padding. The output shrinks with each layer.
- **Same Padding ($P = \lfloor (K-1)/2 \rfloor$)**: Zero-padding is added around the border so output height and width match input dimensions (when $S=1$).

#### 2. Pooling (Max & Average Pooling)
Pooling layers reduce spatial dimensions (width $\times$ height) while preserving channel count:
- **Max Pooling**: Selects the maximum value in each $k \times k$ window. Retains dominant features / sharp edges and provides translation invariance.
- **Average Pooling**: Calculates the arithmetic mean in each $k \times k$ window. Provides smooth downsampling.


In [ ]:
def pooling_scratch(image, pool_size=(2, 2), stride=2, mode='max'):
    """
    Implements Max Pooling and Average Pooling operations from scratch in NumPy.
    """
    if image.ndim == 2:
        image = image[:, :, np.newaxis]
        
    H, W, C = image.shape
    p_H, p_W = pool_size
    
    out_H = (H - p_H) // stride + 1
    out_W = (W - p_W) // stride + 1
    
    output = np.zeros((out_H, out_W, C))
    
    for c in range(C):
        for i in range(out_H):
            for j in range(out_W):
                h_start = i * stride
                h_end = h_start + p_H
                w_start = j * stride
                w_end = w_start + p_W
                
                window = image[h_start:h_end, w_start:w_end, c]
                if mode == 'max':
                    output[i, j, c] = np.max(window)
                elif mode == 'avg':
                    output[i, j, c] = np.mean(window)
                else:
                    raise ValueError("Mode must be 'max' or 'avg'")
                    
    return output.squeeze()

# Test Pooling on sample matrix
test_mat = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [0, 2, 8, 3],
    [4, 1, 7, 9]
], dtype=np.float32)

max_p = pooling_scratch(test_mat, pool_size=(2, 2), stride=2, mode='max')
avg_p = pooling_scratch(test_mat, pool_size=(2, 2), stride=2, mode='avg')

print("Original 4x4 Matrix:\n", test_mat)
print("\nMax Pooling (2x2, stride=2):\n", max_p)
print("\nAverage Pooling (2x2, stride=2):\n", avg_p)


### Visualizing Convolution Filters on Synthetic & Real Images

Let's test our scratch `conv2d_scratch` function by applying classical computer vision kernels (Sobel, Edge Detection, Sharpen, Gaussian Blur, Ridge Detection) to a synthetic test pattern.


In [ ]:
# Create a synthetic image with geometric shapes and sharp edges
def generate_synthetic_image():
    img = np.zeros((128, 128), dtype=np.float32)
    # White square
    img[20:60, 20:60] = 1.0
    # White circle
    y, x = np.ogrid[:128, :128]
    mask = (x - 90)**2 + (y - 90)**2 <= 25**2
    img[mask] = 1.0
    # Diagonal line
    for i in range(128):
        if 15 <= i < 113:
            img[i, 128 - i] = 1.0
            img[min(i+1, 127), 128 - i] = 1.0
    return img

synthetic_img = generate_synthetic_image()

# Define standard convolution kernels
kernels = {
    'Original': None,
    'Sobel Horizontal (X)': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32),
    'Sobel Vertical (Y)': np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32),
    'Laplacian Edge': np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32),
    'Sharpen': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32),
    'Gaussian Blur': np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=np.float32) / 16.0
}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()

for idx, (name, kernel) in enumerate(kernels.items()):
    if name == 'Original':
        res = synthetic_img
    else:
        res = conv2d_scratch(synthetic_img, kernel, stride=1, padding='same')
    
    axes[idx].imshow(res, cmap='gray')
    axes[idx].set_title(name, fontsize=12, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle('Effects of Standard 2D Convolution Kernels', fontsize=16, y=1.02)
plt.show()


---
<a id="2-building-cnn-with-tensorflowkeras"></a>
# 2. Building CNN with TensorFlow/Keras

Now that we understand the math and low-level mechanics of convolution and pooling, let's transition to **TensorFlow / Keras** to build, train, and evaluate CNNs on real image classification tasks.

### Core Keras Layers for Computer Vision
1. **`tf.keras.layers.Conv2D(filters, kernel_size, strides, padding, activation)`**:
   - `filters`: Number of output feature maps (e.g., 32, 64, 128).
   - `kernel_size`: Height and width of the sliding window (e.g., `(3, 3)`).
   - `activation`: Non-linear function (e.g., `'relu'`).
2. **`tf.keras.layers.MaxPooling2D(pool_size, strides)`**:
   - Downsamples spatial dimensions by taking maximum values.
3. **`tf.keras.layers.BatchNormalization()`**:
   - Normalizes layer activations, stabilizing gradient flow and speeding up convergence.
4. **`tf.keras.layers.Dropout(rate)`**:
   - Randomly sets input units to 0 with probability `rate` during training to prevent overfitting.
5. **`tf.keras.layers.Flatten()` / `GlobalAveragePooling2D()`**:
   - Flattens spatial tensor into a 1D vector before feeding into Dense layers.
6. **`tf.keras.layers.Dense(units, activation)`**:
   - Fully connected layer for high-level reasoning and final classification.


### Dataset Preparation: CIFAR-10
The **CIFAR-10** dataset consists of $60,000$ color images of size $32 \times 32 \times 3$ across 10 classes ($50,000$ training images, $10,000$ test images):
`airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`.


In [ ]:
# Load CIFAR-10 dataset
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = datasets.cifar10.load_data()

# Class labels
class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

# Normalize pixel values to [0, 1]
x_train = x_train_raw.astype('float32') / 255.0
x_test = x_test_raw.astype('float32') / 255.0

# Convert labels
y_train = y_train_raw.flatten()
y_test = y_test_raw.flatten()

print(f"Training data shape: {x_train.shape}, Labels shape: {y_train.shape}")
print(f"Test data shape:     {x_test.shape}, Labels shape: {y_test.shape}")

# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i in range(10):
    axes[i].imshow(x_train_raw[i])
    axes[i].set_title(f"Label: {class_names[y_train[i]]}", fontsize=11)
    axes[i].axis('off')

plt.suptitle('CIFAR-10 Sample Images', fontsize=16)
plt.tight_layout()
plt.show()


### Designing a Multi-Block Keras CNN Architecture

We design a 3-block convolutional neural network architecture:
- **Block 1**: Conv2D(32) $\to$ Conv2D(32) $\to$ MaxPool2D $\to$ Dropout(0.25)
- **Block 2**: Conv2D(64) $\to$ Conv2D(64) $\to$ MaxPool2D $\to$ Dropout(0.25)
- **Block 3**: Conv2D(128) $\to$ Conv2D(128) $\to$ MaxPool2D $\to$ Dropout(0.3)
- **Dense Head**: Flatten $\to$ Dense(128) $\to$ Dropout(0.4) $\to$ Dense(10, Softmax)


In [ ]:
def build_custom_cnn(input_shape=(32, 32, 3), num_classes=10):
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=input_shape, name='conv_block1_1'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv_block1_2'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2), name='pool_block1'),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv_block2_1'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv_block2_2'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2), name='pool_block2'),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv_block3_1'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2), name='pool_block3'),
        layers.Dropout(0.3),

        # Fully Connected Head
        layers.Flatten(name='flatten'),
        layers.Dense(128, activation='relu', name='dense_1'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax', name='output')
    ], name='Custom_CIFAR10_CNN')
    
    return model

cnn_model = build_custom_cnn()
cnn_model.summary()


### Model Compilation and Training

We compile our model with the Adam optimizer and sparse categorical crossentropy loss. We train for 10 epochs using a subset/batch split for rapid execution while demonstrating strong convergence.


In [ ]:
cnn_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Use early stopping to prevent overfitting
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Train model (using 10,000 training images and 2,000 validation images for fast demonstration)
train_sub_x, train_sub_y = x_train[:15000], y_train[:15000]
val_sub_x, val_sub_y = x_test[:3000], y_test[:3000]

history = cnn_model.fit(
    train_sub_x, train_sub_y,
    epochs=10,
    batch_size=64,
    validation_data=(val_sub_x, val_sub_y),
    callbacks=[early_stop],
    verbose=1
)


### Training Curves & Model Evaluation

In [ ]:
# Plot Training & Validation Metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, marker='o')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, marker='s')
ax1.set_title('Model Accuracy over Epochs', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

# Loss plot
ax2.plot(history.history['loss'], label='Train Loss', linewidth=2, marker='o')
ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=2, marker='s')
ax2.set_title('Model Loss over Epochs', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Test Set Evaluation
test_loss, test_acc = cnn_model.evaluate(val_sub_x, val_sub_y, verbose=0)
print(f"Validation Test Accuracy: {test_acc * 100:.2f}% | Test Loss: {test_loss:.4f}")

# Predictions & Confusion Matrix
y_pred_probs = cnn_model.predict(val_sub_x, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(val_sub_y, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix on Validation Set', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("\nClassification Report:\n")
print(classification_report(val_sub_y, y_pred, target_names=class_names))


---
<a id="3-popular-cnn-architectures"></a>
# 3. Popular CNN Architectures

Over the past decade, several iconic CNN architectures pushed the state-of-the-art in Computer Vision. In this section, we study and implement three milestone designs:

1. **LeNet-5 (1998)** - The pioneer of modern CNNs.
2. **VGG-16 (2014)** - Standardizing deep modular blocks with $3 \times 3$ convolutions.
3. **ResNet (2015)** - Solving the vanishing gradient problem with residual skip connections.

---
### 1. LeNet-5 Implementation
Introduced by Yann LeCun et al. (1998) for handwritten digit recognition (MNIST), LeNet-5 follows the classic pattern:
`Input (32x32) -> Conv2D(6) -> AvgPool -> Conv2D(16) -> AvgPool -> Dense(120) -> Dense(84) -> Output(10)`


In [ ]:
def build_lenet5(input_shape=(32, 32, 1), num_classes=10):
    model = models.Sequential([
        # C1: Conv Layer
        layers.Conv2D(6, kernel_size=(5, 5), strides=(1, 1), activation='tanh', input_shape=input_shape, name='C1_Conv'),
        # S2: Average Pooling
        layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2), name='S2_AvgPool'),
        # C3: Conv Layer
        layers.Conv2D(16, kernel_size=(5, 5), strides=(1, 1), activation='tanh', name='C3_Conv'),
        # S4: Average Pooling
        layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2), name='S4_AvgPool'),
        # C5: Fully Connected Conv Layer
        layers.Conv2D(120, kernel_size=(5, 5), strides=(1, 1), activation='tanh', name='C5_Conv'),
        layers.Flatten(),
        # F6: Fully Connected Layer
        layers.Dense(84, activation='tanh', name='F6_Dense'),
        # Output Layer
        layers.Dense(num_classes, activation='softmax', name='Output')
    ], name='LeNet-5')
    return model

lenet = build_lenet5()
lenet.summary()


---
### 2. VGG-Style Architecture (Simonyan & Zisserman, 2014)

VGG proved that stacking multiple small $3 \times 3$ filters is superior to using fewer large filters (like $7 \times 7$ or $11 \times 11$):
- **Two stacked $3 \times 3$ filters** have an effective receptive field of $5 \times 5$, but use fewer parameters ($2 \times 3^2 C^2 = 18 C^2$ vs $5^2 C^2 = 25 C^2$) and include two non-linear activations instead of one.
- Channel capacity doubles after each Max Pooling downsampling stage ($64 \to 128 \to 256 \to 512$).


In [ ]:
def build_vgg_block(inputs, filters, num_convs, block_name):
    x = inputs
    for i in range(num_convs):
        x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu', 
                          name=f'{block_name}_conv{i+1}')(x)
        x = layers.BatchNormalization(name=f'{block_name}_bn{i+1}')(x)
    x = layers.MaxPooling2D((2, 2), strides=(2, 2), name=f'{block_name}_pool')(x)
    return x

def build_vgg_style_network(input_shape=(32, 32, 3), num_classes=10):
    inputs = layers.Input(shape=input_shape)
    
    # Block 1
    x = build_vgg_block(inputs, filters=64, num_convs=2, block_name='block1')
    # Block 2
    x = build_vgg_block(x, filters=128, num_convs=2, block_name='block2')
    # Block 3
    x = build_vgg_block(x, filters=256, num_convs=3, block_name='block3')
    
    # Classification Head
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.5, name='dropout')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name='VGG_Style_Network')
    return model

vgg_net = build_vgg_style_network()
vgg_net.summary()


---
### 3. ResNet Architecture & Residual Connections (Skip Connections)

#### The Degradation Problem
As neural networks get deeper, accuracy saturates and then degrades rapidly. This is **not caused by overfitting** (training error increases as well). The root cause is the **vanishing gradient problem**, where backpropagated error signals shrink exponentially as they travel backward through dozens of layers.

#### The Residual Learning Concept (He et al., 2015)
Instead of learning an underlying mapping $H(x)$ directly, residual networks learn a **residual function** $F(x) = H(x) - x$:

$$H(x) = F(x) + x$$

The input $x$ is bypassed around the convolutional layers via a **identity shortcut (skip connection)** and added directly to $F(x)$:

```
           x
           │ ───────┐ (Shortcut Connection)
           ▼        │
      ┌─────────┐   │
      │ Conv2D  │   │
      └────┬────┘   │
           ▼        │
      ┌─────────┐   │
      │ Conv2D  │   │
      └────┬────┘   │
           ▼        │
        F(x)        │
           ├────────┘
           ▼
       F(x) + x  ──> ReLU
```

If $F(x)$ shrinks towards zero, the block degrades to an identity mapping $H(x) = x$, allowing gradients to flow unimpeded directly back to early layers!

#### Dimension Matching
When spatial dimensions decrease (due to stride > 1) or channels increase, the shortcut path uses a **$1 \times 1$ convolution** to match shapes:

$$\text{Output} = F(x) + W_s \cdot x$$


In [ ]:
def residual_block(input_tensor, filters, stride=1, name='res_block'):
    """
    Builds a standard Residual Block with identity shortcut or 1x1 projection shortcut.
    """
    in_channels = input_tensor.shape[-1]
    
    # Main residual path F(x)
    x = layers.Conv2D(filters, (3, 3), strides=stride, padding='same', 
                      use_bias=False, name=f'{name}_conv1')(input_tensor)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.Activation('relu', name=f'{name}_relu1')(x)
    
    x = layers.Conv2D(filters, (3, 3), strides=1, padding='same', 
                      use_bias=False, name=f'{name}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)
    
    # Shortcut path
    shortcut = input_tensor
    if stride != 1 or in_channels != filters:
        # 1x1 Convolution projection to match dimensions
        shortcut = layers.Conv2D(filters, (1, 1), strides=stride, padding='same', 
                                 use_bias=False, name=f'{name}_shortcut_conv')(input_tensor)
        shortcut = layers.BatchNormalization(name=f'{name}_shortcut_bn')(shortcut)
        
    # Element-wise addition F(x) + x
    x = layers.add([x, shortcut], name=f'{name}_add')
    x = layers.Activation('relu', name=f'{name}_out_relu')(x)
    return x

def build_mini_resnet(input_shape=(32, 32, 3), num_classes=10):
    inputs = layers.Input(shape=input_shape)
    
    # Initial Conv
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False, name='init_conv')(inputs)
    x = layers.BatchNormalization(name='init_bn')(x)
    x = layers.Activation('relu')(x)
    
    # Residual Stage 1 (32 channels)
    x = residual_block(x, filters=32, stride=1, name='stage1_block1')
    x = residual_block(x, filters=32, stride=1, name='stage1_block2')
    
    # Residual Stage 2 (64 channels, downsample)
    x = residual_block(x, filters=64, stride=2, name='stage2_block1')
    x = residual_block(x, filters=64, stride=1, name='stage2_block2')
    
    # Residual Stage 3 (128 channels, downsample)
    x = residual_block(x, filters=128, stride=2, name='stage3_block1')
    x = residual_block(x, filters=128, stride=1, name='stage3_block2')
    
    # Output Head
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name='Mini_ResNet')

resnet = build_mini_resnet()
resnet.summary()


---
<a id="4-transfer-learning"></a>
# 4. Transfer Learning

### What is Transfer Learning?
Training deep CNNs from scratch requires massive labeled datasets (million+ images) and days/weeks of compute. **Transfer Learning** leverages knowledge (learned features like edges, textures, shapes) acquired by a model pre-trained on a huge dataset (e.g., ImageNet with 1.4M images and 1,000 classes) and applies it to a target task.

### Strategy Decision Matrix
| Target Dataset Size | Similarity to Source Dataset | Recommended Strategy |
| :--- | :--- | :--- |
| **Small** | **High** (e.g. Natural images) | **Feature Extraction**: Freeze pre-trained base, train top classifier head. |
| **Small** | **Low** (e.g. Medical X-rays) | Train top classifier head + fine-tune top conv layers carefully. |
| **Large** | **High** | Fine-tune whole network with low learning rate. |
| **Large** | **Low** | Train from scratch or fine-tune whole pre-trained network. |

---
### Workflow:
1. **Feature Extraction**:
   - Load pre-trained base network (e.g., `VGG16` or `ResNet50`) without top FC layers (`include_top=False`).
   - Freeze all base layers (`base_model.trainable = False`).
   - Attach custom classification head (`GlobalAveragePooling2D` $\to$ `Dense` $\to$ `Dropout` $\to$ `Output`).
   - Train classifier head.
2. **Fine-Tuning**:
   - Unfreeze top convolutional blocks (`base_model.trainable = True`).
   - Freeze early layers (which contain generic low-level features).
   - Re-compile with a small learning rate (e.g., $10^{-5}$) to avoid destroying pre-trained weights.
   - Train for additional epochs.


In [ ]:
# Load pre-trained VGG16 model without classification head
base_vgg = keras.applications.VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(32, 32, 3)
)

# Step 1: Feature Extraction - Freeze base model
base_vgg.trainable = False

# Attach custom head
inputs = layers.Input(shape=(32, 32, 3))
x = base_vgg(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)

tl_model = models.Model(inputs=inputs, outputs=outputs, name='VGG16_Transfer_Learning')

tl_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("VGG16 Base Frozen Summary:")
tl_model.summary()

# Train Feature Extraction Head (5 epochs)
print("\n--- Phase 1: Feature Extraction Training ---")
history_fe = tl_model.fit(
    train_sub_x, train_sub_y,
    epochs=5,
    batch_size=64,
    validation_data=(val_sub_x, val_sub_y),
    verbose=1
)

# Step 2: Fine-Tuning - Unfreeze upper layers
base_vgg.trainable = True

# Freeze all layers except last convolutional block (block5)
for layer in base_vgg.layers[:-4]:
    layer.trainable = False

tl_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5), # Very low learning rate!
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Phase 2: Fine-Tuning Training ---")
history_ft = tl_model.fit(
    train_sub_x, train_sub_y,
    epochs=5,
    batch_size=64,
    validation_data=(val_sub_x, val_sub_y),
    verbose=1
)


### Transfer Learning Performance Comparison

In [ ]:
fe_acc = history_fe.history['val_accuracy']
ft_acc = history_ft.history['val_accuracy']

plt.figure(figsize=(10, 6))
plt.plot(range(1, 6), fe_acc, 'o--', label='Phase 1: Feature Extraction (Frozen Base)', linewidth=2)
plt.plot(range(6, 11), ft_acc, 's-', label='Phase 2: Fine-Tuning (Unfrozen Block 5)', linewidth=2)
plt.axvline(x=5.5, color='gray', linestyle=':', label='Fine-Tuning Start')

plt.title('Transfer Learning Accuracy: Feature Extraction vs. Fine-Tuning', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()


---
<a id="5-cnn-visualization--interpretability"></a>
# 5. CNN Visualization & Interpretability

Deep neural networks are often criticized as "black boxes". However, CNNs are among the most interpretable deep learning models! In this section, we implement three key visual interpretability techniques:

1. **Intermediate Layer Activations**: Visualizing feature maps across shallow, medium, and deep layers.
2. **Convolution Filters**: Inspecting learned weight matrices in Conv2D layers.
3. **Grad-CAM (Gradient-weighted Class Activation Mapping)**: Generating heatmaps overlaying input images to highlight the exact regions driving predictions.

---
### 1. Visualizing Intermediate Layer Activations

Let's feed a test image through our trained model and extract feature maps at different depths.


In [ ]:
# Pick a sample image from validation set
sample_idx = 7
sample_image = val_sub_x[sample_idx:sample_idx+1]
true_label = class_names[val_sub_y[sample_idx]]

# Display original sample image
plt.figure(figsize=(3, 3))
plt.imshow(val_sub_x[sample_idx])
plt.title(f"True Class: {true_label}", fontsize=12)
plt.axis('off')
plt.show()

# Extract names of convolutional layers
conv_layer_names = [layer.name for layer in cnn_model.layers if 'conv' in layer.name]
print("Convolutional Layers to inspect:", conv_layer_names)

# Build Activation Extraction Model
layer_outputs = [cnn_model.get_layer(name).output for name in conv_layer_names]
activation_model = models.Model(inputs=cnn_model.input, outputs=layer_outputs)

# Get feature map activations
activations = activation_model.predict(sample_image, verbose=0)

# Plot feature maps for Block 1, Block 2, Block 3
for layer_name, feature_map in zip(conv_layer_names[:3], activations[:3]):
    n_features = feature_map.shape[-1]
    size = feature_map.shape[1]
    
    # Display first 16 feature maps in a 2x8 grid
    display_grid = np.zeros((size * 2, size * 8))
    
    for i in range(16):
        row = i // 8
        col = i % 8
        channel_image = feature_map[0, :, :, i]
        # Normalize for display
        channel_image -= channel_image.mean()
        channel_image /= (channel_image.std() + 1e-5)
        channel_image *= 64
        channel_image += 128
        channel_image = np.clip(channel_image, 0, 255).astype('uint8')
        
        display_grid[row * size : (row + 1) * size, col * size : (col + 1) * size] = channel_image
        
    plt.figure(figsize=(16, 4))
    plt.imshow(display_grid, cmap='viridis')
    plt.title(f"Feature Maps from Layer: '{layer_name}' (Shape: {feature_map.shape})", fontsize=14)
    plt.axis('off')
    plt.show()


---
### 2. Visualizing Learned Convolution Filters (Kernels)

Let's extract and visualize the raw weight values of the first Conv2D layer in our network.


In [ ]:
# Get weights of first Conv2D layer
first_conv_layer = cnn_model.get_layer('conv_block1_1')
weights, biases = first_conv_layer.get_weights()

print(f"First Conv Layer Kernel Shape: {weights.shape}")
# Shape: (kernel_h, kernel_w, channels_in, num_filters) e.g., (3, 3, 3, 32)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
axes = axes.ravel()

for i in range(32):
    # Extract i-th filter weight tensor (3, 3, 3)
    filter_w = weights[:, :, :, i]
    # Normalize weights to [0, 1] for visualization
    f_min, f_max = filter_w.min(), filter_w.max()
    filter_w = (filter_w - f_min) / (f_max - f_min + 1e-5)
    
    axes[i].imshow(filter_w)
    axes[i].axis('off')

plt.suptitle('Learned 3x3 RGB Filters in 1st Conv2D Layer', fontsize=16)
plt.tight_layout()
plt.show()


---
### 3. Grad-CAM (Gradient-weighted Class Activation Mapping)

#### Mathematical Overview (Selvaraju et al., 2017)
Grad-CAM calculates gradients of the class score $y^c$ with respect to feature activation maps $A^k$ of the final convolutional layer:

1. **Compute Neuron Importance Weights $w_k^c$**:

$$w_k^c = \frac{1}{Z} \sum_{i} \sum_{j} \frac{\partial y^c}{\partial A_{i,j}^k}$$

2. **Weighted Combination & ReLU Activation**:

$$L_{\text{Grad-CAM}}^c = \text{ReLU}\left( \sum_{k} w_k^c A^k \right)$$

The ReLU filter ensures we highlight only features that have a **positive influence** on the class of interest.


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Generates a Grad-CAM heatmap for a given image and target convolutional layer.
    """
    grad_model = models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    with tf.GradientTape() as tape:
        last_conv_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
        
    # Gradient of target class wrt output of last conv layer
    grads = tape.gradient(class_channel, last_conv_output)
    
    # Vector of mean intensity of gradient over specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Multiply each channel in feature map array by 'how important this channel is'
    last_conv_output = last_conv_output[0]
    heatmap = last_conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    # Apply ReLU to retain positive contributions and normalize [0, 1]
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

# Apply Grad-CAM across multiple sample test images
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx in range(3):
    test_img = val_sub_x[idx:idx+1]
    true_cls = class_names[val_sub_y[idx]]
    
    preds = cnn_model.predict(test_img, verbose=0)
    pred_cls_idx = np.argmax(preds[0])
    pred_cls = class_names[pred_cls_idx]
    confidence = preds[0][pred_cls_idx] * 100
    
    # Generate heatmap from last conv layer
    heatmap = make_gradcam_heatmap(test_img, cnn_model, last_conv_layer_name='conv_block3_1')
    
    # Resize heatmap to match original image size
    heatmap_resized = tf.image.resize(heatmap[..., tf.newaxis], (32, 32)).numpy().squeeze()
    
    # Original Image
    axes[idx, 0].imshow(val_sub_x[idx])
    axes[idx, 0].set_title(f"True: {true_cls}", fontsize=11)
    axes[idx, 0].axis('off')
    
    # Grad-CAM Heatmap
    axes[idx, 1].imshow(heatmap_resized, cmap='jet')
    axes[idx, 1].set_title("Grad-CAM Heatmap", fontsize=11)
    axes[idx, 1].axis('off')
    
    # Superimposed Overlay
    axes[idx, 2].imshow(val_sub_x[idx])
    axes[idx, 2].imshow(heatmap_resized, cmap='jet', alpha=0.4)
    axes[idx, 2].set_title(f"Pred: {pred_cls} ({confidence:.1f}%)", fontsize=11)
    axes[idx, 2].axis('off')

plt.suptitle("Grad-CAM Interpretability on CIFAR-10 Test Samples", fontsize=16)
plt.tight_layout()
plt.show()


---
<a id="6-summary--key-takeaways"></a>
# 6. Summary & Key Takeaways

### Core Concepts Matrix
| Concept / Technique | Key Mechanism | Main Advantage | Typical Use Case |
| :--- | :--- | :--- | :--- |
| **2D Convolution** | Sliding window dot product | Parameter sharing, local receptive fields | Feature extraction in grid-like data |
| **Padding ('same' vs 'valid')** | Zero-border padding | Controls output spatial dimensions | Maintaining image resolution across layers |
| **Pooling (Max / Avg)** | Subsampling windows | Spatial compression, translation invariance | Reducing resolution and compute cost |
| **LeNet-5** | Conv $\to$ Pool $\to$ Conv $\to$ Pool $\to$ FC | Pioneer architecture | Simple OCR / digit recognition |
| **VGG Architecture** | Stacking $3 \times 3$ conv layers | Deep modular feature representation | General image classification baselines |
| **ResNet (Skip Connections)** | Residual learning $F(x) + x$ | Eliminates vanishing gradients in ultra-deep networks | Very deep networks ($50-152+$ layers) |
| **Transfer Learning** | Pre-trained ImageNet weights | Fast training, high performance on small datasets | Domain adaptation & custom tasks |
| **Grad-CAM** | Gradient flow to final conv layer | Visual model interpretability & debugging | Verifying model attention and bias |

---
### Best Practices Checklist for CNN Design
1. **Start Small**: Use small $3 \times 3$ convolutional kernels stacked sequentially rather than large filters.
2. **Normalize Early and Often**: Apply `BatchNormalization` after convolution operations to accelerate training and stabilize gradients.
3. **Use Dropout & Data Augmentation**: Prevent overfitting when dataset size is limited.
4. **Leverage Pre-trained Backbones**: For custom image tasks, default to pre-trained architectures (ResNet, EfficientNet, MobileNet) via Transfer Learning.
5. **Verify Attention with Grad-CAM**: Always inspect class activation maps to confirm your network is learning true object patterns rather than background noise.

---
**Congratulations on completing Module 4!** You now have a solid theoretical and practical foundation in Convolutional Neural Networks, state-of-the-art architectures, transfer learning, and model interpretability.
